# Loan Default Analysis - Home Credit Default Risk

## Objective
Analyze the Home Credit dataset to identify core features that are strong indicators of loan default. This analysis will help understand the key drivers of loan defaults and build predictive models.

**Dataset**: Home Credit Default Risk Competition Data
**Target Variable**: TARGET (0 = Repaid, 1 = Defaulted)

## Section 1: Load and Explore the Dataset

In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the training data
data_path = './data/home-credit-default-risk/application_train.csv'
df = pd.read_csv(data_path)

print("=" * 80)
print("DATASET SHAPE AND BASIC INFO")
print("=" * 80)
print(f"Dataset Shape: {df.shape}")
print(f"\nNumber of Features: {df.shape[1]}")
print(f"Number of Samples: {df.shape[0]}")
print("\nFirst few columns:")
print(df.columns[:10].tolist())

DATASET SHAPE AND BASIC INFO
Dataset Shape: (307511, 122)

Number of Features: 122
Number of Samples: 307511

First few columns:
['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']


In [2]:
%pip install matplotlib seaborn -q

  error: subprocess-exited-with-error
  
  × Building wheel for pillow (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [263 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build/lib.macosx-10.15-universal2-cpython-315/PIL
      copying src/PIL/MpoImagePlugin.py -> build/lib.macosx-10.15-universal2-cpython-315/PIL
      copying src/PIL/ImageMode.py -> build/lib.macosx-10.15-universal2-cpython-315/PIL
      copying src/PIL/PngImagePlugin.py -> build/lib.macosx-10.15-universal2-cpython-315/PIL
      copying src/PIL/AvifImagePlugin.py -> build/lib.macosx-10.15-universal2-cpython-315/PIL
      copying src/PIL/XbmImagePlugin.py -> build/lib.macosx-10.15-universal2-cpython-315/PIL
      copying src/PIL/PcxImagePlugin.py -> build/lib.macosx-10.15-universal2-cpython-315/PIL
      copying src/PIL/SunImagePlugin.py -> build/lib.macosx-10.15-universal2-cpython-315/PIL
      copying src/PIL/ImageFile.py -> build/lib.macosx-10.15

In [8]:
print("\n" + "=" * 80)
print("FIRST FEW ROWS")
print("=" * 80)
print(df.head())

print("\n" + "=" * 80)
print("DATA TYPES AND MISSING VALUES")
print("=" * 80)
missing_info = pd.DataFrame({
    'Column': df.columns,
    'Data_Type': df.dtypes,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_info = missing_info[missing_info['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)
print(missing_info.head(20))



FIRST FEW ROWS
   SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR  \
0      100002       1         Cash loans           M            N   
1      100003       0         Cash loans           F            N   
2      100004       0    Revolving loans           M            Y   
3      100006       0         Cash loans           F            N   
4      100007       0         Cash loans           M            N   

  FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  \
0               Y             0          202500.0    406597.5      24700.5   
1               N             0          270000.0   1293502.5      35698.5   
2               Y             0           67500.0    135000.0       6750.0   
3               Y             0          135000.0    312682.5      29686.5   
4               Y             0          121500.0    513000.0      21865.5   

   ...  FLAG_DOCUMENT_18 FLAG_DOCUMENT_19 FLAG_DOCUMENT_20 FLAG_DOCUMENT_21  \
0  ...               

## Section 2: Data Cleaning and Preprocessing

In [9]:
print("=" * 80)
print("DATA CLEANING SUMMARY")
print("=" * 80)

# Create a copy for analysis
df_clean = df.copy()

# Check for duplicates
duplicates = df_clean.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

# Target variable distribution
print(f"\nTarget Variable (DEFAULT) Distribution:")
print(df_clean['TARGET'].value_counts())
print(f"\nDefault Rate: {df_clean['TARGET'].mean() * 100:.2f}%")

# Identify numerical and categorical columns
numerical_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumerical Columns: {len(numerical_cols)}")
print(f"Categorical Columns: {len(categorical_cols)}")

# Remove ID column from numerical for analysis
if 'SK_ID_CURR' in numerical_cols:
    numerical_cols.remove('SK_ID_CURR')
if 'TARGET' in numerical_cols:
    numerical_cols.remove('TARGET')

print(f"\nColumns for Analysis: {len(numerical_cols)}")


DATA CLEANING SUMMARY
Duplicate rows: 0

Target Variable (DEFAULT) Distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Default Rate: 8.07%

Numerical Columns: 106
Categorical Columns: 16

Columns for Analysis: 104


## Section 3: Exploratory Data Analysis (EDA)

In [10]:
print("=" * 80)
print("SUMMARY STATISTICS FOR NUMERICAL FEATURES")
print("=" * 80)
print(df_clean[numerical_cols[:10]].describe().T)

# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target distribution
target_counts = df_clean['TARGET'].value_counts()
axes[0].bar(target_counts.index, target_counts.values, color=['green', 'red'])
axes[0].set_xlabel('Loan Status')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Variable Distribution\n(0 = Repaid, 1 = Defaulted)')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Repaid (0)', 'Defaulted (1)'])

# Percentage distribution
target_pct = df_clean['TARGET'].value_counts(normalize=True) * 100
axes[1].pie(target_pct.values, labels=['Repaid', 'Defaulted'], autopct='%1.1f%%', 
            colors=['green', 'red'], startangle=90)
axes[1].set_title('Default Rate Distribution')

plt.tight_layout()
plt.show()

print(f"\nDefault Rate: {df_clean['TARGET'].mean() * 100:.2f}%")
print(f"Non-Default Rate: {(1 - df_clean['TARGET'].mean()) * 100:.2f}%")


SUMMARY STATISTICS FOR NUMERICAL FEATURES
                               count           mean            std  \
CNT_CHILDREN                307511.0       0.417052       0.722121   
AMT_INCOME_TOTAL            307511.0  168797.919297  237123.146279   
AMT_CREDIT                  307511.0  599025.999706  402490.776996   
AMT_ANNUITY                 307499.0   27108.573909   14493.737315   
AMT_GOODS_PRICE             307233.0  538396.207429  369446.460540   
REGION_POPULATION_RELATIVE  307511.0       0.020868       0.013831   
DAYS_BIRTH                  307511.0  -16036.995067    4363.988632   
DAYS_EMPLOYED               307511.0   63815.045904  141275.766519   
DAYS_REGISTRATION           307511.0   -4986.120328    3522.886321   
DAYS_ID_PUBLISH             307511.0   -2994.202373    1509.450419   

                                    min            25%           50%  \
CNT_CHILDREN                    0.00000       0.000000       0.00000   
AMT_INCOME_TOTAL            25650.00000  11

NameError: name 'plt' is not defined

## Section 4: Feature Correlation Analysis with Target Variable

In [ ]:
# Calculate correlation with target variable for numerical features
print("=" * 80)
print("CORRELATION WITH TARGET VARIABLE (LOAN DEFAULT)")
print("=" * 80)

correlations = []
for col in numerical_cols:
    # Handle missing values
    valid_data = df_clean[[col, 'TARGET']].dropna()
    if len(valid_data) > 0:
        corr, p_value = pointbiserialr(valid_data['TARGET'], valid_data[col])
        correlations.append({
            'Feature': col,
            'Correlation': corr,
            'Abs_Correlation': abs(corr),
            'P_Value': p_value,
            'Significant': 'Yes' if p_value < 0.05 else 'No'
        })

corr_df = pd.DataFrame(correlations).sort_values('Abs_Correlation', ascending=False)

print("\nTop 20 Features by Absolute Correlation with Default:")
print(corr_df.head(20).to_string())

# Visualize top correlations
fig, ax = plt.subplots(figsize=(12, 8))
top_features = corr_df.head(20)
colors = ['red' if x < 0 else 'green' for x in top_features['Correlation']]
ax.barh(range(len(top_features)), top_features['Correlation'], color=colors)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'])
ax.set_xlabel('Correlation Coefficient')
ax.set_title('Top 20 Features Correlated with Loan Default\n(Red = Negative Corr, Green = Positive Corr)')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.tight_layout()
plt.show()


## Section 5: Identify Key Features for Loan Default

In [ ]:
print("=" * 80)
print("KEY FEATURES ANALYSIS - FEATURES WITH STRONGEST CORRELATION TO DEFAULT")
print("=" * 80)

# Filter significant features with correlation > 0.05
key_features = corr_df[corr_df['Abs_Correlation'] > 0.05].copy()
print(f"\nNumber of Key Features (|Correlation| > 0.05): {len(key_features)}")

print("\n" + "=" * 80)
print("RANKED KEY FEATURES BY CORRELATION STRENGTH")
print("=" * 80)
print(key_features[['Feature', 'Correlation', 'Abs_Correlation', 'P_Value']].to_string())

# Save key features
if len(key_features) > 0:
    key_feature_names = key_features['Feature'].tolist()
    print(f"\n\nTop Feature Drivers of Loan Default:")
    for i, (idx, row) in enumerate(key_features.head(15).iterrows(), 1):
        direction = "↑ increases default risk" if row['Correlation'] > 0 else "↓ decreases default risk"
        print(f"{i:2d}. {row['Feature']:35s} | Corr: {row['Correlation']:+.4f} | {direction}")


## Section 6: Statistical Analysis and Insights

In [ ]:
print("=" * 80)
print("COMPARATIVE ANALYSIS: DEFAULTERS VS NON-DEFAULTERS")
print("=" * 80)

# Analyze top 5 features
top_5_features = key_features.head(5)['Feature'].tolist()

for feature in top_5_features:
    print(f"\n{feature}:")
    print("-" * 60)
    defaulters = df_clean[df_clean['TARGET'] == 1][feature].dropna()
    non_defaulters = df_clean[df_clean['TARGET'] == 0][feature].dropna()
    
    print(f"  Defaulters (n={len(defaulters)}):")
    print(f"    Mean: {defaulters.mean():.4f}, Median: {defaulters.median():.4f}, Std: {defaulters.std():.4f}")
    print(f"  Non-Defaulters (n={len(non_defaulters)}):")
    print(f"    Mean: {non_defaulters.mean():.4f}, Median: {non_defaulters.median():.4f}, Std: {non_defaulters.std():.4f}")
    print(f"  Difference (Default - Non-Default): {defaulters.mean() - non_defaulters.mean():.4f}")
    
    # T-test
    t_stat, p_val = stats.ttest_ind(defaulters, non_defaulters, nan_policy='omit')
    print(f"  T-test p-value: {p_val:.4e} {'*' if p_val < 0.05 else ''}")


## Section 7: Visualize Feature Relationships

In [ ]:
# Visualize top 6 features with box plots
top_6_features = key_features.head(6)['Feature'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, feature in enumerate(top_6_features):
    # Prepare data
    data_default = df_clean[df_clean['TARGET'] == 1][feature].dropna()
    data_non_default = df_clean[df_clean['TARGET'] == 0][feature].dropna()
    
    # Create box plot
    axes[idx].boxplot([data_non_default, data_default], 
                       labels=['Repaid (0)', 'Defaulted (1)'],
                       patch_artist=True)
    axes[idx].set_ylabel(feature)
    axes[idx].set_title(f'{feature}\n(Correlation: {corr_df[corr_df["Feature"]==feature]["Correlation"].values[0]:.4f})')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Top 6 Features Associated with Loan Default\nBox Plot Comparison', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Box plot visualization: Comparing feature distributions between defaulters and non-defaulters")


In [ ]:
# Distribution plots for top 4 features
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.ravel()

for idx, feature in enumerate(key_features.head(4)['Feature'].tolist()):
    data_default = df_clean[df_clean['TARGET'] == 1][feature].dropna()
    data_non_default = df_clean[df_clean['TARGET'] == 0][feature].dropna()
    
    axes[idx].hist(data_non_default, bins=50, alpha=0.6, label='Repaid (0)', color='green')
    axes[idx].hist(data_default, bins=50, alpha=0.6, label='Defaulted (1)', color='red')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'Distribution: {feature}')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Distribution of Top 4 Features by Loan Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Distribution plots showing how features differ between repaid and defaulted loans")


In [5]:
print("=" * 80)
print("LOADING AND MERGING BUREAU DATA")
print("=" * 80)

# Load bureau data (external credit history from other institutions)
bureau_path = './data/bureau.csv'
bureau = pd.read_csv(bureau_path)

print(f"\nBureau Data Shape: {bureau.shape}")
print(f"Bureau Columns: {list(bureau.columns)}")
print(f"\nBureau Data Sample:")
print(bureau.head())

# Check the key linking column
print(f"\nUnique clients in bureau: {bureau['SK_ID_CURR'].nunique()}")
print(f"Unique clients in application_train: {df['SK_ID_CURR'].nunique()}")
print(f"Common clients: {len(set(bureau['SK_ID_CURR']).intersection(set(df['SK_ID_CURR'])))}")

# Aggregate bureau data by SK_ID_CURR since there are multiple records per client
# (clients may have multiple credit accounts)
print("\nAggregating bureau data by client...")
bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'DAYS_CREDIT': 'min',  # Earliest credit
    'CREDIT_DAY_OVERDUE': ['max', 'sum'],  # Max overdue days and total
    'AMT_CREDIT_MAX_OVERDUE': 'max',  # Max overdue amount
    'CNT_CREDIT_PROLONG': 'sum',  # Total prolongations
    'AMT_CREDIT_SUM': 'sum',  # Total credit amount
    'AMT_CREDIT_SUM_DEBT': 'sum',  # Total debt
    'AMT_CREDIT_SUM_LIMIT': 'sum',  # Total limit
    'AMT_CREDIT_SUM_OVERDUE': 'sum',  # Total overdue
    'SK_ID_BUREAU': 'count'  # Number of credit accounts
}).reset_index()

# Flatten column names
bureau_agg.columns = ['SK_ID_CURR', 'BUREAU_DAYS_CREDIT_MIN', 'BUREAU_CREDIT_DAY_OVERDUE_MAX', 
                      'BUREAU_CREDIT_DAY_OVERDUE_SUM', 'BUREAU_AMT_CREDIT_MAX_OVERDUE', 
                      'BUREAU_CNT_CREDIT_PROLONG', 'BUREAU_AMT_CREDIT_SUM', 'BUREAU_AMT_CREDIT_SUM_DEBT',
                      'BUREAU_AMT_CREDIT_SUM_LIMIT', 'BUREAU_AMT_CREDIT_SUM_OVERDUE', 'BUREAU_NUM_ACCOUNTS']

print(f"\nAggregated Bureau Shape: {bureau_agg.shape}")
print(f"Bureau Aggregated Sample:")
print(bureau_agg.head())

# Merge bureau data with application data
print("\nMerging bureau data with application data...")
df_merged = df.merge(bureau_agg, on='SK_ID_CURR', how='left')

print(f"Merged Dataset Shape: {df_merged.shape}")
print(f"Original Shape: {df.shape}")
print(f"New Features Added: {df_merged.shape[1] - df.shape[1]}")
print(f"\nMissing values in new features:")
print(df_merged.iloc[:, -11:].isnull().sum())

# Use the merged dataset for further analysis
df = df_merged
print("\n✓ Merged dataset is now the active dataframe")

LOADING AND MERGING BUREAU DATA

Bureau Data Shape: (1716428, 17)
Bureau Columns: ['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']

Bureau Data Sample:
   SK_ID_CURR  SK_ID_BUREAU CREDIT_ACTIVE CREDIT_CURRENCY  DAYS_CREDIT  \
0      215354       5714462        Closed      currency 1         -497   
1      215354       5714463        Active      currency 1         -208   
2      215354       5714464        Active      currency 1         -203   
3      215354       5714465        Active      currency 1         -203   
4      215354       5714466        Active      currency 1         -629   

   CREDIT_DAY_OVERDUE  DAYS_CREDIT_ENDDATE  DAYS_ENDDATE_FACT  \
0                   0               -153.0      

## Section 4: Merge with External Bureau Data

In [ ]:
# Correlation heatmap for top features
top_10_features = key_features.head(10)['Feature'].tolist()
correlation_matrix = df_clean[top_10_features + ['TARGET']].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix: Top 10 Key Features vs Target\n(Loan Default Prediction)', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Correlation heatmap showing relationships between top features and loan default")


## Summary: Key Insights and Conclusions

In [1]:
print("=" * 80)
print("FINAL SUMMARY AND KEY INSIGHTS")
print("=" * 80)

summary_insights = f"""
1. DATASET OVERVIEW:
   - Total Records: {len(df_clean):,}
   - Total Features: {df_clean.shape[1]}
   - Default Rate: {df_clean['TARGET'].mean() * 100:.2f}%
   - Class Imbalance: This is a highly imbalanced dataset

2. KEY FEATURES DRIVING LOAN DEFAULTS:
   The top features most correlated with loan defaults are:
   
"""

print(summary_insights)

# Print top 10 key features
print("   Rank | Feature Name                           | Correlation | Direction")
print("   " + "-" * 77)
for i, (_, row) in enumerate(key_features.head(10).iterrows(), 1):
    direction = "Increases risk" if row['Correlation'] > 0 else "Decreases risk"
    print(f"   {i:2d}.  | {row['Feature']:35s} | {row['Correlation']:+.4f}      | {direction}")

print("\n3. INTERPRETATION:")
print("   - Features with POSITIVE correlation: Increase the likelihood of default")
print("   - Features with NEGATIVE correlation: Decrease the likelihood of default")

print("\n4. RECOMMENDATIONS FOR MODEL BUILDING:")
print("   - Focus on the top 10-15 features for initial model development")
print("   - Handle missing values appropriately (imputation or removal)")
print("   - Consider feature scaling for distance-based algorithms")
print("   - Use class balancing techniques (SMOTE, class weights, etc.)")
print("   - Perform feature engineering to create interaction terms")
print("   - Consider non-linear relationships using tree-based models")

print("\n5. BUSINESS INSIGHTS:")
print("   - Borrowers with certain characteristics have significantly higher default rates")
print("   - The identified features can be used for risk assessment and credit decisions")
print("   - Regular monitoring of these features is recommended for portfolio management")

print("\n" + "=" * 80)


FINAL SUMMARY AND KEY INSIGHTS


NameError: name 'df_clean' is not defined